# Runner AI: training the three networks on Google Colab

**Before you start:** in the menu **Runtime → Change runtime type** choose **T4 GPU** and save. The GPU speeds up the network updates; the game itself runs on the CPU.

**Bring your recordings:** play the app on your computer first. Every run is saved in `data/recordings/`. Zip the whole project folder *with* those recordings and upload it here. Without human recordings, the notebook falls back to the bot's practice runs.

At the end, download `models.zip` and extract it into the project folder on your computer.

## 1. Upload the project
Choose the project zip file (with your recordings inside `data/recordings/`).

In [ ]:
from google.colab import files
uploaded = files.upload()
zip_name = next(iter(uploaded))
!unzip -q -o "$zip_name"
import glob, os
# the project folder: the one with src/make_bot_demos.py (works for the project zip or the whole repository)
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath(sorted(glob.glob('**/src/make_bot_demos.py', recursive=True), key=len)[0]))))
print('Project folder:', os.getcwd())
!ls data/recordings | head
!echo "recordings: $(ls data/recordings/*.json 2>/dev/null | wc -l)"

## 2. Check the GPU
At the start of each training run you should see `Device: cuda`.

In [ ]:
!nvidia-smi -L
import torch
print('PyTorch', torch.__version__, '| GPU:', torch.cuda.is_available())

## 3. (Only if there are no human recordings) practice runs from the bot
Skip this if you uploaded your own recordings: training always prefers human runs when there are any.

In [ ]:
import glob
if not [f for f in glob.glob('data/recordings/*.json') if not f.split('/')[-1].startswith('bot_')]:
    !python -m src.make_bot_demos
else:
    print('Human recordings found - using them.')

## 4. Imitation learning (behavioral cloning)
The network learns to press what the players pressed. A few minutes.

In [ ]:
!python -m src.train_bc
from IPython.display import Image, display
display(Image('reports/imitation_curves.png'))

## 5. (Optional) How much play does imitation need?
Trains on 10%, 25%, 50% and 100% of the recordings and plots the distance against minutes of play. Does not change the saved model.

In [ ]:
# !python -m src.train_bc --curve
# display(Image('reports/imitation_data_curve.png'))

## 6. DQN: reinforcement learning from scratch
The longest step: about 1.9 million ticks of play. The validation line shows the distance on 16 tracks the network never trains on.

In [ ]:
!python -m src.train_dqn
display(Image('reports/dqn_curves.png'))

## 7. Combo: learn from the recordings first, then practice (DQfD)
Half the practice of plain DQN. Does starting from human play make up for it?

In [ ]:
!python -m src.train_dqn --combo
display(Image('reports/combo_curves.png'))

## 8. Evaluation
All players on 50 new tracks (up to 5 minutes each), compared with the recorded human runs.

In [ ]:
!python -m src.evaluate
for f in ['eval_distances', 'eval_survival', 'learning_curves']:
    display(Image(f'reports/{f}.png'))

## 9. Download the networks and reports
* Extract `models.zip` into the project folder on your computer (it creates `models/imitation.pt`, `dqn.pt` and `combo.pt`)
* Extract `reports.zip` the same way

If the browser blocks multiple downloads, download from the files pane (the folder icon on the left).

In [ ]:
!zip -q -r models.zip models
!zip -q -r reports.zip reports
files.download('models.zip')
files.download('reports.zip')

## 10. (Optional) Export for the web page (GitHub Pages)
Packs the trained networks into `docs/model/`, so the page in `docs/` can race them in the browser. Extract `docs_model.zip` into the project folder on your computer, then publish (see the README).

In [ ]:
!python -m src.export_web
!zip -q -r docs_model.zip docs/model
files.download('docs_model.zip')